# Module 03: Evaluation -- Teaching AI to Grade Its Own Work

**Estimated time: 45-60 minutes**

---

## Learning Objectives

By the end of this notebook, you will be able to:
- Explain the LLM-as-judge pattern and when it applies
- Interpret evaluation dimension scores
- Understand what each dimension measures and how to improve it
- Design a regression test to catch silent quality degradation

## The Evaluation Problem

Building an AI system is the easy part. **Knowing if it works is hard.**

For security triage, we need to evaluate:
- Severity accuracy (categorical -- easy to check programmatically)
- MITRE technique accuracy (categorical -- easy to check)
- Reasoning quality (free-text -- hard to automate with rules)
- Actionability of recommendations (free-text -- hard to automate)
- Completeness (did the agent address all indicators?)

**LLM-as-judge** solves the free-text evaluation problem.
Use a second LLM to evaluate the first LLM's output against a rubric.

Key insight: **Judging quality is easier than generating quality.**
A smaller, cheaper model can consistently evaluate whether a decision
has good reasoning -- even if it couldn't produce that reasoning itself.

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath('..'))

from src.pipeline.mock import MOCK_DECISIONS
print('Loaded ' + str(len(MOCK_DECISIONS)) + ' mock triage decisions')

## The 5 Evaluation Dimensions

We evaluate each triage decision across 5 dimensions (scored 1-5):

| Dimension | Score 1 | Score 5 |
|---|---|---|
| **Severity Accuracy** | Wrong direction | Exact match |
| **ATT&CK Mapping** | No technique identified | All techniques correct |
| **Reasoning Quality** | Unsupported/vague | Grounded, step-by-step, coherent |
| **Actionability** | Generic/vague recommendations | Specific, ordered, directly applicable |
| **Completeness** | Major gaps in coverage | Every indicator addressed |

Why 5 dimensions instead of one accuracy number?
- A decision can be correct on severity but have terrible reasoning
- A decision can have excellent reasoning but recommend useless actions
- Separate dimensions tell you **where** to improve the system

In [ ]:
# Check severity accuracy against expert baselines
EXPERT_BASELINES = {
    'ALERT-2024-001': {'severity': 'CRITICAL', 'mitre_techniques': ['T1566.001', 'T1059.001']},
    'ALERT-2024-002': {'severity': 'CRITICAL', 'mitre_techniques': ['T1048']},
    'ALERT-2024-003': {'severity': 'CRITICAL', 'mitre_techniques': ['T1110.001', 'T1078']},
    'ALERT-2024-004': {'severity': 'HIGH',     'mitre_techniques': ['T1566.001']},
    'ALERT-2024-005': {'severity': 'CRITICAL', 'mitre_techniques': ['T1078.004']},
}

print('=== SEVERITY ACCURACY vs. EXPERT BASELINES ===')
print('Alert ID             AI Decision  Expert Baseline  Match')
print('-' * 60)

matches = 0
for alert_id, baseline in EXPERT_BASELINES.items():
    ai_sev = MOCK_DECISIONS[alert_id]['severity']
    exp_sev = baseline['severity']
    match = 'MATCH' if ai_sev == exp_sev else 'MISS'
    if ai_sev == exp_sev:
        matches += 1
    print(alert_id + '  ' + ai_sev.ljust(12) + exp_sev.ljust(17) + match)

pct = int(matches / len(EXPERT_BASELINES) * 100)
print()
print('Severity Accuracy: ' + str(matches) + '/' + str(len(EXPERT_BASELINES)) + ' = ' + str(pct) + '%')

In [ ]:
# Check MITRE technique accuracy
print('=== MITRE TECHNIQUE ACCURACY ===')
print()

for alert_id, baseline in EXPERT_BASELINES.items():
    ai_techniques = set(MOCK_DECISIONS[alert_id]['mitre_techniques'])
    expert_techniques = set(baseline['mitre_techniques'])

    overlap = ai_techniques & expert_techniques
    precision = len(overlap) / len(ai_techniques) if ai_techniques else 0
    recall = len(overlap) / len(expert_techniques) if expert_techniques else 0

    print(alert_id + ':')
    print('  AI found:    ' + str(ai_techniques))
    print('  Expert says: ' + str(expert_techniques))
    print('  Precision: ' + str(int(precision*100)) + '%  Recall: ' + str(int(recall*100)) + '%')
    print()

## Why 80% Is a Strong Starting Point

Mock mode achieves 80% severity accuracy (4/5 exact matches).
With a real LLM, this typically improves to 85-92%.

**Is 80% good enough?** It depends on the workflow:

```
BAD: AI replaces analyst --> 80% right, 20% wrong escalations

GOOD: AI triages, analyst reviews AI-flagged items
  --> AI processes 1,000 alerts, flags 50 as Critical/High
  --> Analyst reviews 50 (instead of 1,000)
  --> ~40 of 50 are genuinely actionable (80% accuracy)
  --> Analyst catches the 10 misclassified items
  --> Net: analyst focuses on 20x fewer items, misses nothing
```

The goal isn't to eliminate human judgment.
It's to make human judgment faster and less fatiguing.

## Exercises

### Beginner
Read through `data/sample_alerts.json` and write your own triage decision for each of the 5 alerts
(severity + MITRE technique + 2-3 recommended actions). Compare to the AI output.
Where do you agree? Where do you disagree? Why?

### Intermediate
Design a 6th evaluation dimension: **False Positive Risk**.
- What does a score of 1 look like? (Agent never considers FP possibility)
- What does a score of 5 look like? (Agent explicitly reasons about FP indicators)
- Write a rubric description that a judge LLM could apply consistently.

### Advanced
Build a regression test harness:
1. Run evaluation against mock decisions and record all dimension scores as a baseline
2. Write a pytest test that fails if any dimension drops more than 10 points from baseline
3. Deliberately modify a triage prompt to degrade quality, verify the test catches it
4. Integrate the test into `.github/workflows/ci.yml`

---

## Curriculum Complete

You've completed all 4 modules:
- **Module 00**: Why agentic AI for security -- the problem and mental model
- **Module 01**: ReAct agents -- the reasoning loop, tools, structured extraction
- **Module 02**: LLM providers -- running for free, factory pattern, cost realities
- **Module 03**: Evaluation -- LLM-as-judge, dimensions, regression testing

You now have the foundation to build, extend, and evaluate agentic AI systems for security.

**What to do next:**
- Work through the Intermediate and Advanced exercises in each module
- Try running with a real LLM (Ollama or Groq -- both free)
- Fork this repo and add a new alert type from a domain you know well
- Share what you build